In [ ]:
import wandb
api = wandb.Api()
run = api.run("/vlm_rl/new_rl_hm3d_v1_full/runs/y71bb2l0") #231
# run = api.run("/vlm_rl/rl_baseline_sft/runs/qxn9wjhy") #sft
# run = api.run("/vlm_rl/rl_baseline/runs/ab9u6sa8") #base
# run = api.run("/vlm_rl/rpp_ke_ablation/runs/wyrtn7bk") #sparse

history = run.history(samples=5000)
true_success = (history['fpg_trigger_count'] < 1) * history['success']#*(history['sup/sum_spguard_trigger_count']<2)
true_spl = (history['fpg_trigger_count'] < 1) * history['spl']#*(history['sup/sum_spguard_trigger_count']<2)
history['true_success'] = true_success
history['true_spl'] = true_spl
print(f"num rows: {len(history['spl'])}")
print(f"true success rate: {history['true_success'].mean()}")
print(f"true spl: {history['true_spl'].mean()}")

num rows: 1932
true success rate: 0.7400310719834283
true spl: 0.44260460257030987


In [27]:
local_results_dir = "/Projects/SG_VLN_HumanData/spatial_training/dump/results/bfloat16fa2_rpp_baseline_111"
from glob import glob
import numpy as np
import pandas as pd
history = pd.DataFrame()
for path in glob(local_results_dir+"/worker_*/result*"): #jsonl results files
    df = pd.read_json(path,lines=True)
    true_success = (df['fpg_trigger_count'] < 1) * df['success']#*(df['sup/sum_spguard_trigger_count']<1)
    true_spl = (df['fpg_trigger_count'] < 1) * df['spl']#*(df['sup/sum_spguard_trigger_count']<1)
    df['true_success'] = true_success
    df['true_spl'] = true_spl
    history = pd.concat([history,df],ignore_index=True)
    print(f"num rows: {len(df['spl'])}")
    print(f"true success rate: {df['true_success'].mean()}")
    print(f"true spl: {df['true_spl'].mean()}")

num rows: 98
true success rate: 0.6836734693877551
true spl: 0.4073467605883269
num rows: 96
true success rate: 0.75
true spl: 0.40156153278500345


In [4]:
import json
with open("../oracle_durations_sample400.json","r") as f:
    oracle_durations = json.load(f)

In [6]:
filtered_history = history[history['episode_label'].map(lambda x: x in oracle_durations.keys())]
filtered_history['oracle_duration'] = filtered_history['episode_label'].map(lambda x: oracle_durations[x])*2
print(filtered_history['true_success'].mean())
print(filtered_history['true_spl'].mean())

0.7298701298701299
0.4408382675357187


/tmp/ipykernel_374478/1904592063.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_history['oracle_duration'] = filtered_history['episode_label'].map(lambda x: oracle_durations[x])*2


In [13]:
start=200
end=500
slice_history = filtered_history[(filtered_history['oracle_duration'] >= start) & (filtered_history['oracle_duration'] < end)]
slice_history

,_runtime,sup/min_vlm_latency,episode_details,sup/max_sim_latency,distance_to_goal,episode_id,collision_rate,fpg_trigger_count,_timestamp,goal,...,success,worker_pid,throughput,spl,n_steps,soft_spl,episode_label,true_success,true_spl,oracle_duration
3,129.447212,0.064340,None,0.083500,0.148684,82,0.015113,42.0,1.769637e+09,toilet,...,1.0,2962631.0,4.205032,0.409465,397.0,0.406747,6s7QHgap2fW_82,0.0,0.000000,262
6,156.265760,0.066178,None,0.066188,15.026011,2,0.000000,31.0,1.769637e+09,bed,...,0.0,2962633.0,3.906204,0.000000,500.0,0.145651,cvZr5TUy5C5_2,0.0,0.000000,296
15,305.393234,0.067566,None,0.063088,13.701196,73,0.020000,69.0,1.769637e+09,bed,...,0.0,2962633.0,3.694174,0.000000,500.0,0.236255,cvZr5TUy5C5_73,0.0,0.000000,282
30,459.135795,0.065480,None,0.043379,19.687822,5,0.030000,98.0,1.769637e+09,plant,...,0.0,2962630.0,4.323796,0.000000,500.0,0.090263,TEEsavR23oF_5,0.0,0.000000,284
91,1234.910076,0.069182,None,0.081080,9.307038,91,0.100000,89.0,1.769638e+09,toilet,...,0.0,2962633.0,4.443999,0.000000,500.0,0.192429,qyAac8rV8Zk_91,0.0,0.000000,204
102,1320.619254,0.064217,None,0.127233,0.011564,68,0.005376,0.0,1.769638e+09,tv_monitor,...,1.0,2962632.0,3.568808,0.691216,186.0,0.690755,6s7QHgap2fW_68,1.0,0.691216,202
127,1637.175954,0.065918,None,0.060393,18.043690,73,0.004000,56.0,1.769638e+09,tv_monitor,...,0.0,2962630.0,4.024353,0.000000,500.0,0.049599,6s7QHgap2fW_73,0.0,0.000000,262
138,1698.678096,0.067573,None,0.050548,0.068308,45,0.004115,0.0,1.769639e+09,toilet,...,1.0,2962631.0,5.793315,0.527119,243.0,0.525139,qyAac8rV8Zk_45,1.0,0.527119,200
175,2127.379507,0.065201,None,0.039599,0.441082,72,0.002959,11.0,1.769639e+09,tv_monitor,...,1.0,2962630.0,5.517653,0.483015,338.0,0.473842,5cdEh9F2hJL_72,0.0,0.000000,270
223,2722.859743,0.067389,None,0.114975,21.985579,59,0.000000,55.0,1.769640e+09,tv_monitor,...,0.0,2962631.0,4.288422,0.000000,500.0,0.062086,cvZr5TUy5C5_59,0.0,0.000000,280


In [7]:
duration_slices = [0,50,200,500]
# compute success rate and spl for each duration slice
num_list = []
for i in range(len(duration_slices)-1):
    start = duration_slices[i]
    end = duration_slices[i+1]
    slice_history = filtered_history[(filtered_history['oracle_duration'] >= start) & (filtered_history['oracle_duration'] < end)]
    if len(slice_history) == 0:
        continue
    slice_true_success = slice_history['true_success'].mean()
    slice_true_spl = slice_history['true_spl'].mean()
    num_list.extend([slice_true_success,slice_true_spl])
    # print(f"Duration {start}-{end}: True Success Rate: {slice_true_success}, True SPL: {slice_true_spl}")
    
num_str=" & ".join([f"{x*100:.1f}" for x in num_list])
print(num_str)

86.2 & 53.2 & 69.9 & 41.7 & 15.4 & 9.4
